In [1]:
import requests
import cloudscraper
from bs4 import BeautifulSoup
import pandas as pd
import re

donnees = []
scraper = cloudscraper.create_scraper()

for j in range(1,30):
    if j==1:  
        page = requests.get("https://www.etreproprio.com/annonces/thf.ld69#list")
    else:
        page = requests.get("https://www.etreproprio.com/annonces/thf.ld69.odd.g"+str(j)+"#list")
        
    soup = BeautifulSoup(page.content, "html.parser")
    temp = soup.find("div", class_='ep-search-list-wrapper').find_all("a")
    
    liens = []
    for i in range(len(temp)):
        if "https://www.etreproprio.com/immobilier-" in temp[i]['href']:
            liens.append(temp[i]['href'])
    
    for lien in liens:
        try:
            page_annonce = scraper.get(lien)
            soup_ = BeautifulSoup(page_annonce.content, 'lxml')

            try:
                # "Vente Appartement 75 m² Paris-14e"
                titre = soup_.find("title").text.strip()
            except:
                titre = None

            try:
                # "Appartement 75 m² à Paris-14e"
                description_h1 = soup_.find("h1", class_='annonce-immobilier').text.strip()
            except:
                description_h1 = None

            try:
                # Extrait la ville depuis le titre : "Vente Appartement 75 m² Paris-14e" → "Paris-14e"
                ville = soup_.find("title").text.strip().split("²")[-1].strip()
            except:
                ville = None

            try:
                prix = soup_.find("div", class_='ep-price').text.strip()
                prix = int(prix.replace(" ", "").replace("\xa0", "").replace("€", ""))
            except:
                prix = None

            try:
                surface = soup_.find("div", class_='ep-area').text.strip()
                surface = int(surface.replace("\xa0", "").replace("m²", "").strip())
            except:
                surface = None

            try:
                type_bien = titre.split(" ")[1]  # le 2ème mot après "Vente"
            except:
                type_bien = None

            try:
                desc_texte = soup_.find("div", class_='ep-desc-truncated').text.strip()
            except:
                desc_texte = None
            
            try:
                etage = None
                if desc_texte:
                # Cas 1 — chiffres
                    etage_match = re.search(r'(\d+)\s*(?:[eèéê][mr]?e?|°)?\s*[ée]tage', desc_texte, re.IGNORECASE)
                    if etage_match:
                        etage = int(etage_match.group(1))
                # Cas 2 — lettres
                    else:
                        etages_lettres = {
                            "premier": 1, "première": 1,
                            "deuxième": 2, "deuxieme": 2,
                            "troisième": 3, "troisieme": 3,
                            "quatrième": 4, "quatrieme": 4,
                            "cinquième": 5, "cinquieme": 5,
                            "sixième": 6, "sixieme": 6,
                            "septième": 7, "septieme": 7,
                            "huitième": 8, "huitieme": 8,
                            "neuvième": 9, "neuvieme": 9,
                            "dixième": 10, "dixieme": 10,
                        }
                        for mot, num in etages_lettres.items():
                            if mot in desc_texte.lower():
                                etage = num
                                break
                # Cas 3 — rez-de-chaussée
                    if etage is None:
                        rdc_patterns = ["rez-de-chaussée", "rez de chaussée","rez-de-chaussee", "rdc", "r.d.c"]
                        if any(p in desc_texte.lower() for p in rdc_patterns):
                            etage = 0
            except:
                etage = None

            try:
                dpe = soup_.find("div", class_='dpe-diagram').find("div", class_='selected').text.strip()
            except:
                dpe = None

            try:
                pieces = soup_.find("div", class_='ep-room').text.strip()
                pieces = int(pieces.replace("pièces", "").replace("pièce", "").strip())
            except:
                pieces = None

            print(f"Titre: {titre} | Ville: {ville} | Prix: {prix} | Surface: {surface} | Type: {type_bien} | Etage: {etage} | Pièces: {pieces} | DPE: {dpe}")

            donnees.append({
                "titre": titre,
                "description": description_h1,
                "ville": ville,
                "type_bien": type_bien,
                "etage": etage,
                "prix": prix,
                "surface": surface,
                "pieces": pieces,
                "DPE": dpe,
                "full_description": desc_texte,
                "lien": lien
            })

        except Exception as e:
            print(f"Erreur sur {lien} : {e}")
            continue

# Création du DataFrame
df = pd.DataFrame(donnees)
df["prix"] = pd.to_numeric(df["prix"], errors="coerce")
df["surface"] = pd.to_numeric(df["surface"], errors="coerce")
df["pieces"] = pd.to_numeric(df["pieces"], errors="coerce").astype("Int64")
df["etage"] = pd.to_numeric(df["etage"], errors="coerce").astype("Int64")

print(df)

Titre: Vente Maison 147 m² Saint-Genis-Les-Ollieres | Ville: Saint-Genis-Les-Ollieres | Prix: 475000 | Surface: None | Type: Maison | Etage: 4 | Pièces: 6 | DPE: C
Titre: Vente Appartement 119 m² Lyon-6e | Ville: Lyon-6e | Prix: 555000 | Surface: 119 | Type: Appartement | Etage: 3 | Pièces: 5 | DPE: C
Titre: Vente Appartement 23 m² Lyon-2e | Ville: Lyon-2e | Prix: 141474 | Surface: 23 | Type: Appartement | Etage: 5 | Pièces: 1 | DPE: D
Titre: Vente Appartement 60 m² Lyon-4e | Ville: Lyon-4e | Prix: 227000 | Surface: 60 | Type: Appartement | Etage: 1 | Pièces: 3 | DPE: D
Titre: Vente Appartement 121 m² Villeurbanne | Ville: Villeurbanne | Prix: 430000 | Surface: 121 | Type: Appartement | Etage: None | Pièces: 5 | DPE: None
Titre: Vente Appartement 61 m² Caluire-Et-Cuire | Ville: Caluire-Et-Cuire | Prix: 305000 | Surface: 61 | Type: Appartement | Etage: 0 | Pièces: 3 | DPE: D
Titre: Vente Appartement 18 m² Lyon-5e | Ville: Lyon-5e | Prix: 144000 | Surface: 18 | Type: Appartement | Etage:

In [2]:
df

,titre,description,ville,type_bien,etage,prix,surface,pieces,DPE,full_description,lien
0,Vente Maison 147 m² Saint-Genis-Les-Ollieres,"Pavillon de 147m2 avec piscine, 4 chambres, gr...",Saint-Genis-Les-Ollieres,Maison,4,475000,NaN,6,C,L’agence Valdor a le plaisir de vous présenter...,https://www.etreproprio.com/immobilier-2422233...
1,Vente Appartement 119 m² Lyon-6e,Appartement 119 m² à Lyon-6e,Lyon-6e,Appartement,3,555000,119.0,5,C,"Lyon 6, Halles Paul Bocuse, appartement famili...",https://www.etreproprio.com/immobilier-2529239...
2,Vente Appartement 23 m² Lyon-2e,Investissement en residence étudiante,Lyon-2e,Appartement,5,141474,23.0,1,D,Simplicité et Tranquillité de votre investisse...,https://www.etreproprio.com/immobilier-2528410...
3,Vente Appartement 60 m² Lyon-4e,Appartement avec vue sur le Rhône ! Proche Têt...,Lyon-4e,Appartement,1,227000,60.0,3,D,LYON 4e – Cours d'Herbouville face au Parc de ...,https://www.etreproprio.com/immobilier-2528532...
4,Vente Appartement 121 m² Villeurbanne,Appartement 121 m² à Villeurbanne,Villeurbanne,Appartement,<NA>,430000,121.0,5,None,VILLEURBANNE - PLACE MARENGO\n\nAppartement du...,https://www.etreproprio.com/immobilier-2507612...
...,...,...,...,...,...,...,...,...,...,...,...
575,Vente Appartement 95 m² Lyon-5e,T4 rénové de 95 m² avec vue,Lyon-5e,Appartement,5,329000,95.0,4,E,Tassin / Limite Lyon 5. Dans résidence sécuris...,https://www.etreproprio.com/immobilier-2526522...
576,Vente Appartement 50 m² Vaulx-En-Velin,Appartement 50 m² à Vaulx-En-Velin,Vaulx-En-Velin,Appartement,<NA>,150000,50.0,2,D,Novea Immobilier a le plaisir de vous présente...,https://www.etreproprio.com/immobilier-2526010...
577,Vente Appartement 18 m² Lyon-7e,Investissement en residence étudiante,Lyon-7e,Appartement,2,102398,18.0,1,D,Un investissement patrimonial et sécurisé Inv...,https://www.etreproprio.com/immobilier-2525707...
578,Vente Appartement 32 m² Lyon-1e,Studio dernier étage avec chambre séparée - mo...,Lyon-1e,Appartement,1,224000,32.0,2,None,"Au cœur d’un secteur recherché, découvrez ce c...",https://www.etreproprio.com/immobilier-2525925...


In [3]:
df.isna().sum()

titre                 0
description           0
ville                 0
type_bien             0
etage               252
prix                  0
surface             160
pieces                5
DPE                  55
full_description      7
lien                  0
dtype: int64

In [4]:
extracted = df["titre"].str.extract(r'(\d+(?:[.,]\d+)?)\s*m\S*')[0].str.replace(",", ".", regex=False).astype(float)
extracted

0      147.0
1      119.0
2       23.0
3       60.0
4      121.0
       ...  
575     95.0
576     50.0
577     18.0
578     32.0
579    196.0
Name: 0, Length: 580, dtype: float64

In [5]:
extracted

0      147.0
1      119.0
2       23.0
3       60.0
4      121.0
       ...  
575     95.0
576     50.0
577     18.0
578     32.0
579    196.0
Name: 0, Length: 580, dtype: float64

In [6]:
df["surface"] = df["surface"].fillna(extracted)

In [7]:
df.isna().sum()

titre                 0
description           0
ville                 0
type_bien             0
etage               252
prix                  0
surface               3
pieces                5
DPE                  55
full_description      7
lien                  0
dtype: int64

In [8]:
print(df.loc[df["surface"].isna(), "titre"].iloc[0])
print(df.loc[df["surface"].isna(), "description"].iloc[0])
print(df.loc[df["surface"].isna(), "full_description"].iloc[0])

Vente Appartement  Sainte-Foy-Les-Lyon
Appartement Sainte Foy Les Lyon 2 pièce(s)
Dans une copropriété calme et sécurisée, un appartement de type 2 de 51.6m² Carrez avec un jardin privatif de 40m² et une terrasse de 15m². Le bien est vendu avec une cave. Cette résidence a été construite au début des année 2000 et  se situe chemin du Vallon sur la charmante commune de Sainte Foy Les Lyon, à proximité de l'arrêt de bus de la ligne 49 (Chavril) en direction de Lyon Perrache. L'appartement est composé d'un hall d'entrée avec un dressing, un séjour pouvant être ouvert sur la cuisine équipée, une chambre avec placard murale, un Wc séparé et une salle de bains. Le Chauffage est individuel au gaz et les fenêtres sont en double vitrage PVC avec les stores roulants et manuels. Les honoraires sont à la charge du vendeur. Possibilité d'acquérir un garage fermé en sus (20 000 euros TTC) Pour une visite, Jimmy BRILLAUD au 06x10x40x32x79 Copropriété de 60 lots - dont 22 lots habitation.


In [9]:
extracted_new = df["full_description"].str.extract(r'(\d+(?:[.,]\d+)?)\s*m\S*')[0].str.replace(",", ".", regex=False).astype(float)
extracted_new

0      150.00
1      119.20
2       23.15
3       57.75
4         NaN
        ...  
575     95.00
576     51.00
577     18.10
578     33.00
579    800.00
Name: 0, Length: 580, dtype: float64

In [10]:
df["surface"] = df["surface"].fillna(extracted_new)

In [11]:
df[df["titre"]=="Vente Appartement  Sainte-Foy-Les-Lyon"]

,titre,description,ville,type_bien,etage,prix,surface,pieces,DPE,full_description,lien
226,Vente Appartement Sainte-Foy-Les-Lyon,Appartement Sainte Foy Les Lyon 2 pièce(s),Vente Appartement Sainte-Foy-Les-Lyon,Appartement,<NA>,240000,51.6,2,D,"Dans une copropriété calme et sécurisée, un ap...",https://www.etreproprio.com/immobilier-2528632...


In [12]:
df.isna().sum()

titre                 0
description           0
ville                 0
type_bien             0
etage               252
prix                  0
surface               0
pieces                5
DPE                  55
full_description      7
lien                  0
dtype: int64

In [13]:
df.to_csv("annonces_rhone.csv", index=False, encoding="utf-8-sig")

In [14]:
df["full_description"][0]

'L’agence Valdor a le plaisir de vous présenter une maison de standing (construction en 2001) d’environ 150 m² habitables, idéalement située dans un quartier résidentiel recherché et particulièrement paisible de Saint-Genis-les-Ollières.\n\r\nCette maison est située au sein d’un lotissement entièrement clos et sécurisé, bénéficiant d’un système de vidéosurveillance et d’un portail électrique à l’entrée. Anciennement le bâtiment central d’un club de vacances (en effet il y avait un village vacances à Saint-Genis les Ollières il y a plus de 10 ans), il a conservé son caractère unique et ses volumes généreux. Aujourd’hui, la piscine ainsi que les espaces extérieurs attenants sont privatifs, parfaitement clos, à l’abri des regards et idéalement exposés au sud-ouest.\n\r\nDès l’entrée, vous serez immédiatement séduit par les volumes exceptionnels de la pièce de vie de plus de 60 m². Cet espace accueille une cuisine ouverte, moderne et design, baignée de lumière grâce aux larges baies vitrée

In [15]:
df

,titre,description,ville,type_bien,etage,prix,surface,pieces,DPE,full_description,lien
0,Vente Maison 147 m² Saint-Genis-Les-Ollieres,"Pavillon de 147m2 avec piscine, 4 chambres, gr...",Saint-Genis-Les-Ollieres,Maison,4,475000,147.0,6,C,L’agence Valdor a le plaisir de vous présenter...,https://www.etreproprio.com/immobilier-2422233...
1,Vente Appartement 119 m² Lyon-6e,Appartement 119 m² à Lyon-6e,Lyon-6e,Appartement,3,555000,119.0,5,C,"Lyon 6, Halles Paul Bocuse, appartement famili...",https://www.etreproprio.com/immobilier-2529239...
2,Vente Appartement 23 m² Lyon-2e,Investissement en residence étudiante,Lyon-2e,Appartement,5,141474,23.0,1,D,Simplicité et Tranquillité de votre investisse...,https://www.etreproprio.com/immobilier-2528410...
3,Vente Appartement 60 m² Lyon-4e,Appartement avec vue sur le Rhône ! Proche Têt...,Lyon-4e,Appartement,1,227000,60.0,3,D,LYON 4e – Cours d'Herbouville face au Parc de ...,https://www.etreproprio.com/immobilier-2528532...
4,Vente Appartement 121 m² Villeurbanne,Appartement 121 m² à Villeurbanne,Villeurbanne,Appartement,<NA>,430000,121.0,5,None,VILLEURBANNE - PLACE MARENGO\n\nAppartement du...,https://www.etreproprio.com/immobilier-2507612...
...,...,...,...,...,...,...,...,...,...,...,...
575,Vente Appartement 95 m² Lyon-5e,T4 rénové de 95 m² avec vue,Lyon-5e,Appartement,5,329000,95.0,4,E,Tassin / Limite Lyon 5. Dans résidence sécuris...,https://www.etreproprio.com/immobilier-2526522...
576,Vente Appartement 50 m² Vaulx-En-Velin,Appartement 50 m² à Vaulx-En-Velin,Vaulx-En-Velin,Appartement,<NA>,150000,50.0,2,D,Novea Immobilier a le plaisir de vous présente...,https://www.etreproprio.com/immobilier-2526010...
577,Vente Appartement 18 m² Lyon-7e,Investissement en residence étudiante,Lyon-7e,Appartement,2,102398,18.0,1,D,Un investissement patrimonial et sécurisé Inv...,https://www.etreproprio.com/immobilier-2525707...
578,Vente Appartement 32 m² Lyon-1e,Studio dernier étage avec chambre séparée - mo...,Lyon-1e,Appartement,1,224000,32.0,2,None,"Au cœur d’un secteur recherché, découvrez ce c...",https://www.etreproprio.com/immobilier-2525925...
